In [1]:
# build_faiss_index.py
import os, json, time
from pathlib import Path
from typing import List, Dict, Any

# --- small-GPU friendly defaults ---
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from tqdm import tqdm
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# -------- CONFIG --------
JSONL_PATH  = "data/chunks.md.jsonl"
INDEX_DIR   = "faiss_index"
MODEL_NAME  = "BAAI/bge-small-en-v1.5"   # fast & solid
GPU_BATCH   = 64                         # safe for ~4GB VRAM; raise if you can
CPU_BATCH   = 32
MAX_SEQ_LEN = 256                        # reduce if you still OOM (e.g., 192)

# -------- IO --------
def load_jsonl_as_documents(path: str) -> List[Document]:
    docs: List[Document] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec: Dict[str, Any] = json.loads(line)
            docs.append(
                Document(
                    page_content=str(rec.get("text", "")),
                    metadata=rec.get("metadata", {}) or {},
                )
            )
    return docs

# -------- Embeddings helpers --------
def make_embedder(device: str) -> HuggingFaceEmbeddings:
    """
    Build a HuggingFaceEmbeddings that runs on the given device and
    uses memory-friendly settings. We DO NOT pass torch_dtype to avoid
    the init error you hit.
    """
    emb = HuggingFaceEmbeddings(
        model_name=MODEL_NAME,
        model_kwargs={"device": device},
        encode_kwargs={"batch_size": GPU_BATCH if device == "cuda" else CPU_BATCH,
                       "normalize_embeddings": True},
    )
    # Cap max sequence length to reduce VRAM
    try:
        st_model = emb.client  # SentenceTransformer
        st_model.max_seq_length = MAX_SEQ_LEN
    except Exception:
        pass
    return emb

def embed_in_batches(emb: HuggingFaceEmbeddings, texts: List[str], batch_size: int) -> List[List[float]]:
    vecs: List[List[float]] = []
    using_cuda = torch.cuda.is_available() and emb.client.device.type == "cuda"
    for i in tqdm(range(0, len(texts), batch_size),
                  desc=f"Embedding ({'GPU' if using_cuda else 'CPU'})"):
        batch = texts[i:i+batch_size]
        vecs.extend(emb.embed_documents(batch))
        if using_cuda:
            torch.cuda.empty_cache()
    return vecs

def robust_embed_all(texts: List[str]) -> (List[List[float]], HuggingFaceEmbeddings):
    """
    Try GPU first with conservative settings; if OOM or any GPU error,
    fall back to CPU automatically. Return (vectors, query_embedder).
    The returned embedder is what we also pass into FAISS.
    """
    if torch.cuda.is_available():
        try:
            emb_gpu = make_embedder("cuda")
            vecs = embed_in_batches(emb_gpu, texts, batch_size=GPU_BATCH)
            return vecs, emb_gpu
        except RuntimeError as e:
            if "CUDA out of memory" in str(e):
                print("\n[GPU OOM] Falling back to CPU…")
            else:
                print(f"\n[GPU error] {e}\nFalling back to CPU…")
        except Exception as e:
            print(f"\n[GPU init error] {e}\nFalling back to CPU…")

    emb_cpu = make_embedder("cpu")
    vecs = embed_in_batches(emb_cpu, texts, batch_size=CPU_BATCH)
    return vecs, emb_cpu



In [ ]:
# -------- Main --------
if __name__ == "__main__":
    print("CUDA available:", torch.cuda.is_available())

    # 1) Load chunks
    chunks = load_jsonl_as_documents(JSONL_PATH)
    if not chunks:
        raise SystemExit(f"No chunks found in {JSONL_PATH}")
    print(f"Loaded {len(chunks)} chunks")

    texts = [d.page_content for d in chunks]
    metas = [d.metadata for d in chunks]

    # 2) Embed (GPU if possible, else CPU)
    t0 = time.time()
    vectors, query_emb = robust_embed_all(texts)
    print(f"Embedded {len(vectors)} vectors in {time.time()-t0:.1f}s")

    # 3) Build FAISS from precomputed embeddings
    pairs = list(zip(texts, vectors))
    # IMPORTANT: pass the embedding object here (this fixes your TypeError)
    vs = FAISS.from_embeddings(pairs, embedding=query_emb, metadatas=metas)

    # 4) Save index
    Path(INDEX_DIR).mkdir(parents=True, exist_ok=True)
    vs.save_local(INDEX_DIR)
    print(f"✅ Saved FAISS index to: {INDEX_DIR}")

    # 5) Reload & test search with the SAME embedder config you’ll use at query-time
    vs = FAISS.load_local(INDEX_DIR, embeddings=query_emb, allow_dangerous_deserialization=True)

    q = "how to install foo"
    print(f"\nTop matches for: {q!r}")
    for d, score in vs.similarity_search_with_score(q, k=3):
        snippet = d.page_content[:160].replace("\n", " ")
        print(f"{score:.4f} | {d.metadata.get('path','')} | {snippet}")

        # scores represnting distance, so lower distance is better


CUDA available: True
Loaded 2095 chunks


C:\Users\Sachin Kumar Jha\AppData\Local\Temp\ipykernel_16372\4231442017.py:47: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  emb = HuggingFaceEmbeddings(
c:\Users\Sachin Kumar Jha\.conda\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Sachin Kumar Jha\.conda\envs\rag\Lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pyto

Embedded 2095 vectors in 32.9s
✅ Saved FAISS index to: faiss_index

Top matches for: 'how to install foo'
0.5849 | Subscription-manager / #fdo | = ["fips=1"]   EOF   RUN dnf install -y crypto-policies-scripts &amp;&amp; update-crypto-policies --no-reload --set FIPS   2.  Build the &lt;image&gt; image by 
0.6021 | systemd services RUN systemctl enable httpd sshd &amp;&amp; \ systemctl disable telnetd &amp;&amp; \ systemctl mask rcpbind / Procedure | put:/output \   registry.redhat.io/rhel9/bootc-image-builder:latest \   - --type iso \ - --config /config.toml \ - quay.io/ &lt;namespace&gt; / &lt;image&gt; : 
0.6033 | firewall-cmd --add-port=8081/tcp --permanent # firewall-cmd --add-port=8083/tcp --permanent # systemctl restart firewalld / Procedure | ## Procedure   1.  Create a Containerfile, for example:   FROM registry.redhat.io/rhel10/rhel-bootc:latest RUN dnf install -y fdo-init fdo-client RUN systemctl 


In [6]:
len(pairs[0][1])

384

In [7]:
len(pairs[1][1])

384

In [8]:
q = "RHEL"
print(f"\nTop matches for: {q!r}")
for d, score in vs.similarity_search_with_score(q, k=3):
    snippet = d.page_content[:160].replace("\n", " ")
    print(f"{score:.4f} | {d.metadata.get('path','')} | {snippet}")


Top matches for: 'RHEL'
0.5350 | Red Hat Enterprise Linux 10 Composing, installing, and managing RHEL for | g, and managing RHEL , and managing RHEL and managing RHEL and managing RHEL nd managing RHEL d managing RHEL managing RHEL managing RHEL anaging RHEL naging RH
0.5626 | 2.1. IMAGE MODE FOR RHEL | ## 2.1. IMAGE MODE FOR RHEL   Image mode for Red Hat Enterprise Linux (RHEL) is a deployment method that uses a container-native approach to build, deploy, and 
0.5664 | systemd services RUN systemctl enable httpd sshd &amp;&amp; \ systemctl disable telnetd &amp;&amp; \ systemctl mask rcpbind / 2.5.3. Using bootc-image-builder to create RHEL 10.0 disk images | longer supports composing customized RHEL rpm-ostree images optimized for Edge. To create new RHEL images for Edge environments as part of RHEL 10, you must use


In [11]:
results = vs.similarity_search_with_score("how to install foo", k=3)

def faiss_score_to_cos(score):
    # score is FAISS's squared L2 distance for unit-normalized vectors
    # return cosine similarity in [0,1]
    return 1.0 - (score / 4.0)

for d, dist2 in results:
    sim01 = faiss_score_to_cos(dist2)
    x = d.page_content[:160].replace('\n',' ')
    print(f"sim={sim01:.3f} | path={d.metadata.get('path','')} | {x}")


sim=0.854 | path=Subscription-manager / #fdo | = ["fips=1"]   EOF   RUN dnf install -y crypto-policies-scripts &amp;&amp; update-crypto-policies --no-reload --set FIPS   2.  Build the &lt;image&gt; image by 
sim=0.849 | path=systemd services RUN systemctl enable httpd sshd &amp;&amp; \ systemctl disable telnetd &amp;&amp; \ systemctl mask rcpbind / Procedure | put:/output \   registry.redhat.io/rhel9/bootc-image-builder:latest \   - --type iso \ - --config /config.toml \ - quay.io/ &lt;namespace&gt; / &lt;image&gt; : 
sim=0.849 | path=firewall-cmd --add-port=8081/tcp --permanent # firewall-cmd --add-port=8083/tcp --permanent # systemctl restart firewalld / Procedure | ## Procedure   1.  Create a Containerfile, for example:   FROM registry.redhat.io/rhel10/rhel-bootc:latest RUN dnf install -y fdo-init fdo-client RUN systemctl 


In [16]:
print("num docstore entries:", len(vs.docstore._dict))
first_ids = vs.index_to_docstore_id
for i, doc_id in first_ids.items():
    doc = vs.docstore._dict[doc_id]
    print(f"[{i}] row_id={i} -> doc_id={doc_id}")
    print("   meta:", doc.metadata)
    print("   text:", (doc.page_content[:120].replace("\n"," ") + "..."))

num docstore entries: 2095
[0] row_id=0 -> doc_id=037e38f2-1e8f-4c0f-ae50-b5bf24841d57
   meta: {'source': 'output'}
   text: <!-- image !-- image -- image - image image image mage age ge e --> --> -> >...
[1] row_id=1 -> doc_id=84e8dd21-82e3-450d-bec3-fba01670405f
   meta: {'source': 'output', 'section': 'Red Hat Enterprise Linux 10 Composing, installing, and managing RHEL for', 'path': 'Red Hat Enterprise Linux 10 Composing, installing, and managing RHEL for'}
   text: ## Red Hat Enterprise Linux 10 Composing, installing, and managing RHEL # Red Hat Enterprise Linux 10 Composing, install...
[2] row_id=2 -> doc_id=557a58c2-d967-413d-97bb-3e23bde487f1
   meta: {'source': 'output', 'section': 'Red Hat Enterprise Linux 10 Composing, installing, and managing RHEL for', 'path': 'Red Hat Enterprise Linux 10 Composing, installing, and managing RHEL for'}
   text: se Linux 10 Composing, installing, and managing RHEL e Linux 10 Composing, installing, and managing RHEL Linux 10 Compos...
[3] ro

In [15]:
type(first_ids)

dict